# FinOps, Performance e Escalabilidade

Este notebook documenta e demonstra as decisões de arquitetura adotadas para otimizar armazenamento, processamento e custos da Literacy Data Pipeline.

O objetivo é aplicar princípios de FinOps à solução implementada, relacionando eficiência técnica e uso responsável de recursos computacionais.

A análise considera:

- otimização do armazenamento;
- redução de processamento desnecessário;
- processamento incremental;
- otimizações Spark/PySpark;
- separação entre armazenamento e processamento;
- escalabilidade da arquitetura;
- limitações do ambiente Databricks Free Edition.

## Configuração

Definição dos parâmetros utilizados nas análises de FinOps e eficiência da arquitetura.

In [0]:
# Configura o bucket utilizado pelo projeto.

BUCKET_NAME = "literacy-data-pipeline-tcf2-480749290106-us-east-1-an"

BRONZE_PATH = f"s3://{BUCKET_NAME}/bronze/"
SILVER_PATH = f"s3://{BUCKET_NAME}/silver/"
GOLD_PATH = f"s3://{BUCKET_NAME}/gold/"

print(f"Bucket monitorado: {BUCKET_NAME}")

## 1. Decisões FinOps adotadas

A arquitetura foi construída considerando eficiência desde as primeiras camadas da pipeline.

As principais decisões relacionadas a FinOps são:

- utilização do Amazon S3 como Data Lake, desacoplando armazenamento e processamento;
- preservação dos arquivos originais em CSV apenas na camada Bronze;
- utilização de Parquet nas camadas Silver e Gold;
- criação de tabelas analíticas na Gold para evitar o recálculo frequente de transformações;
- processamento incremental no fluxo Streaming;
- utilização de checkpoints para evitar reprocessamento de eventos;
- execução do Structured Streaming com `availableNow`, evitando manter processamento ativo sem necessidade;
- validação da granularidade e das chaves antes de joins;
- separação das responsabilidades entre Bronze, Silver e Gold.

## 2. Otimização de armazenamento

Os arquivos brutos são preservados em CSV na camada Bronze para garantir rastreabilidade e fidelidade às fontes originais.

Após o tratamento, os dados são armazenados em Parquet. O formato colunar permite leitura seletiva de colunas e é mais adequado ao processamento analítico realizado pelo Spark.

Para demonstrar o impacto dessa decisão, é comparado o volume armazenado de um dataset na Bronze e sua respectiva versão tratada na Silver.

In [0]:
# Compara o volume físico armazenado do dataset avaliacao_alunos entre a Bronze em CSV e a Silver em Parquet.

BRONZE_ALUNOS_PATH = (
    f"s3://{BUCKET_NAME}/bronze/avaliacao_alunos.csv"
)

SILVER_ALUNOS_PATH = (
    f"s3://{BUCKET_NAME}/silver/avaliacao_alunos/"
)

arquivo_bronze = dbutils.fs.ls(BRONZE_ALUNOS_PATH)[0]

arquivos_silver = dbutils.fs.ls(SILVER_ALUNOS_PATH)

tamanho_bronze_bytes = arquivo_bronze.size

tamanho_silver_bytes = sum(
    arquivo.size
    for arquivo in arquivos_silver
    if arquivo.name.endswith(".parquet")
)

tamanho_bronze_mb = round(
    tamanho_bronze_bytes / (1024 * 1024), 2
)

tamanho_silver_mb = round(
    tamanho_silver_bytes / (1024 * 1024), 2
)

reducao_percentual = round(
    (
        (tamanho_bronze_bytes - tamanho_silver_bytes)
        / tamanho_bronze_bytes
    ) * 100,
    2
)

print(f"Bronze CSV: {tamanho_bronze_mb} MB")
print(f"Silver Parquet: {tamanho_silver_mb} MB")
print(f"Redução de armazenamento: {reducao_percentual}%")

### 2.1 Impacto do formato de armazenamento

A comparação entre a versão bruta em CSV e a versão tratada em Parquet permite observar o impacto da escolha do formato sobre o volume armazenado.

A redução de armazenamento contribui para princípios de FinOps porque pode diminuir custos de armazenamento, transferência e leitura dos dados. Além disso, o formato colunar permite que mecanismos analíticos leiam apenas as colunas necessárias para cada processamento.

## 3. Otimização de processamento e consultas

As otimizações de consulta do projeto são aplicadas por meio da API DataFrame do Spark/PySpark.

Como as camadas Silver e Gold utilizam Parquet, o Spark pode aplicar otimizações como leitura seletiva de colunas e filtros antes de carregar todo o conjunto necessário para a análise.

Nesta etapa é analisado o plano físico de uma consulta sobre a Silver para demonstrar essas otimizações.

In [0]:
# Carrega o dataset de alunos em Parquet para demonstrar otimizações de leitura e processamento realizadas pelo Spark.

df_alunos_silver = spark.read.parquet(
    f"{SILVER_PATH}avaliacao_alunos/"
)

print(
    f"[OK] Base carregada: "
    f"{df_alunos_silver.count()} registros"
)

In [0]:
# Aplica projeção de colunas e filtro antes da agregação, reduzindo o volume de dados necessário para o processamento analítico.

from pyspark.sql import functions as F

df_consulta_otimizada = (
    df_alunos_silver
    .select(
        "ano",
        "id_municipio",
        "preenchimento_caderno",
        "alfabetizado",
        "proficiencia"
    )
    .filter(
        F.col("preenchimento_caderno") == 1
    )
    .groupBy("ano", "id_municipio")
    .agg(
        F.count("*").alias("alunos_avaliados"),
        F.round(
            F.avg("proficiencia"), 2
        ).alias("proficiencia_media")
    )
)

In [0]:
# Exibe o plano de execução utilizado pelo Spark para a consulta otimizada.

df_consulta_otimizada.explain("formatted")

In [0]:
# Exibe uma amostra do resultado produzido pela consulta otimizada.

display(
    df_consulta_otimizada
    .orderBy("ano", "id_municipio")
    .limit(20)
)

## 4. Processamento incremental

O fluxo Streaming foi implementado com checkpoint e processamento incremental para evitar o reprocessamento de eventos já consumidos.

Essa abordagem reduz processamento desnecessário e melhora a eficiência operacional, pois apenas novos dados precisam ser processados a cada execução.

In [0]:
# Define os caminhos do Streaming para demonstrar o uso de processamento incremental.

BRONZE_STREAM_PATH = (
    f"s3://{BUCKET_NAME}/bronze/streaming/avaliacao_alunos/"
)

SILVER_STREAM_PATH = (
    f"s3://{BUCKET_NAME}/silver/streaming/avaliacao_alunos/"
)

BRONZE_CHECKPOINT_PATH = (
    f"s3://{BUCKET_NAME}/checkpoints/bronze_streaming/"
)

SILVER_CHECKPOINT_PATH = (
    f"s3://{BUCKET_NAME}/checkpoints/silver_streaming/"
)

In [0]:
# Verifica a existência dos checkpoints utilizados pelo Structured Streaming.

checkpoints = {
    "bronze_streaming": BRONZE_CHECKPOINT_PATH,
    "silver_streaming": SILVER_CHECKPOINT_PATH
}

for nome, caminho in checkpoints.items():
    try:
        arquivos = dbutils.fs.ls(caminho)

        print(
            f"[OK] {nome}: checkpoint encontrado "
            f"({len(arquivos)} item(ns))"
        )

    except Exception as erro:
        print(f"[ERRO] {nome}: checkpoint não encontrado - {erro}")

In [0]:
# Compara os eventos persistidos nas camadas Bronze e Silver Streaming.

df_bronze_stream = spark.read.parquet(BRONZE_STREAM_PATH)
df_silver_stream = spark.read.parquet(SILVER_STREAM_PATH)

registros_bronze_stream = df_bronze_stream.count()
registros_silver_stream = df_silver_stream.count()

print(f"Bronze Streaming: {registros_bronze_stream} registros")
print(f"Silver Streaming: {registros_silver_stream} registros")

if registros_bronze_stream == registros_silver_stream:
    print(
        "[OK] Fluxo incremental sincronizado entre Bronze e Silver."
    )

## 5. Escalabilidade da arquitetura

A arquitetura foi desenhada para permitir crescimento de volume de dados sem exigir mudanças estruturais nas camadas da pipeline.

O Amazon S3 atua como armazenamento desacoplado da capacidade computacional. Isso permite aumentar o volume armazenado independentemente dos recursos utilizados pelo Databricks e pelo Apache Spark.

Em um ambiente produtivo, a solução poderia escalar por meio de:

- aumento ou redução dos recursos computacionais do Spark conforme o volume processado;
- processamento distribuído de datasets maiores;
- paralelização das transformações entre diferentes partições;
- particionamento dos dados por ano, região ou outra chave relevante;
- processamento incremental para evitar reprocessamento completo;
- inclusão de novas fontes mantendo a Bronze histórica preservada;
- aumento do armazenamento no S3 sem necessidade de redimensionar a camada de processamento.

In [0]:
# Demonstra que os dados permanecem persistidos no Amazon S3 independentemente da sessão de processamento do Databricks.

CAMADAS = {
    "bronze": BRONZE_PATH,
    "silver": SILVER_PATH,
    "gold": GOLD_PATH
}

for camada, caminho in CAMADAS.items():
    itens = dbutils.fs.ls(caminho)

    print(
        f"[OK] {camada.upper()}: "
        f"{len(itens)} item(ns) disponíveis no S3"
    )

### 5.1 Separação entre armazenamento e processamento

O Data Lake permanece no Amazon S3 enquanto o Databricks é responsável pelo processamento.

Essa separação permite que recursos computacionais sejam iniciados, redimensionados ou encerrados sem perda dos dados persistidos, contribuindo tanto para escalabilidade quanto para controle de custos.

Em cenários de maior volume, a capacidade computacional pode ser ajustada conforme a demanda sem necessidade de migrar ou replicar todo o armazenamento.

## 6. Limitações do ambiente acadêmico

O projeto foi desenvolvido utilizando o Databricks Free Edition, adequado para implementação e demonstração da arquitetura proposta, porém com limitações em relação a um ambiente produtivo.

Por esse motivo, não foram realizadas análises reais de custo computacional ou dimensionamento de clusters.

Em um ambiente produtivo, o monitoramento FinOps poderia incluir:

- custo por execução da pipeline;
- duração dos jobs;
- consumo de recursos computacionais;
- volume de dados lidos e gravados;
- custo de armazenamento no Amazon S3;
- comparação de custo entre processamento completo e incremental;
- acompanhamento de utilização dos recursos;
- alertas de orçamento e anomalias de custo.

## 7. Recomendações para ambiente produtivo

Para uma evolução da solução em ambiente produtivo, recomenda-se:

- dimensionar os recursos computacionais de acordo com o volume e a frequência das cargas;
- utilizar autoscaling quando aplicável;
- desligar recursos computacionais quando não estiverem em uso;
- adotar processamento incremental sempre que a fonte permitir;
- avaliar estratégias de particionamento conforme o crescimento dos dados;
- monitorar tamanho e quantidade de arquivos no Data Lake;
- acompanhar métricas de custo e consumo;
- definir budgets e alertas de custo na AWS;
- manter dados históricos no S3 e utilizar recursos computacionais apenas durante o processamento;
- revisar periodicamente consultas e planos de execução Spark.

## 8. Tecnologias e decisões arquiteturais

### Tecnologias utilizadas

| Tecnologia | Utilização | Justificativa |
|---|---|---|
| AWS S3 | Data Lake | Armazenamento escalável e desacoplado do processamento |
| Databricks | Processamento | Integração com Spark, notebooks e Structured Streaming |
| Apache Spark / PySpark | Batch e Streaming | Processamento distribuído e API única para os dois paradigmas |
| Unity Catalog | Governança | Organização e controle do acesso aos dados |
| Parquet | Silver e Gold | Formato colunar otimizado para armazenamento e análise |
| Git/GitHub | Versionamento | Histórico, branches e revisão por Pull Request |
| Python | Implementação | Integração direta com PySpark e ecossistema de dados |

### Principais trade-offs

#### Batch vs Streaming

O processamento Batch foi utilizado para os datasets históricos e consolidados, enquanto o Structured Streaming foi implementado para representar a chegada incremental de novos eventos.

A arquitetura híbrida permite combinar eficiência para grandes cargas históricas com menor latência para novos dados.

#### Data Lake vs Data Warehouse

Foi adotado um Data Lake no Amazon S3 utilizando arquitetura Medalhão.

Essa escolha permite preservar os dados brutos na Bronze, manter dados tratados na Silver e disponibilizar informações analíticas na Gold sem exigir um Data Warehouse adicional para o escopo atual.

Um Data Warehouse poderia ser incorporado futuramente caso existisse necessidade de alto volume de consultas SQL concorrentes ou consumo por múltiplas ferramentas de BI.

#### Custo vs Performance

A arquitetura prioriza equilíbrio entre custo e desempenho.

A Bronze mantém os arquivos CSV originais, enquanto Silver e Gold utilizam Parquet para reduzir armazenamento e melhorar a eficiência das leituras analíticas.

O processamento incremental, checkpoints, seleção de colunas, validação prévia de joins e materialização de tabelas Gold reduzem processamento repetido.

Essas otimizações consomem armazenamento adicional ao manter múltiplas camadas, mas diminuem a necessidade de reprocessamento e melhoram o desempenho das consultas.

%md
## 9. Conclusão

As decisões de arquitetura adotadas no projeto demonstram que FinOps não se limita ao acompanhamento financeiro da infraestrutura, mas também envolve escolhas técnicas que reduzem desperdício de recursos.

A utilização de Parquet nas camadas tratadas, processamento incremental com checkpoint, otimizações do Spark e separação entre armazenamento e processamento contribuem para uma arquitetura mais eficiente e preparada para crescimento.

Embora o ambiente acadêmico não permita mensurar todos os custos de uma operação produtiva, a solução implementada incorpora princípios que permitem evoluir o projeto para um cenário de maior escala com controle de desempenho e custos.